In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

## Full-field flash – exploratory analysis

This notebook explores how model neurons respond to a **full-field luminance flash**
that steps from a uniform gray background to a brighter level.

Three things are investigated:

1. **Stimulus construction** – build a 12-frame achromatic flash stimulus
   (gray pre-flash context, then bright flash).
2. **Contrast sweep** – vary flash contrast from 0 to 1 and plot predicted
   responses for a handful of example neurons.
3. **Gaze sweep** – fix contrast at a high value and shift the simulated gaze
   position; show how the predicted responses of the same neurons change.

### Stimulus layout

| Frames | Content | Valid predictions? |
|--------|---------|-------------------|
| 0–2    | gray (mean_lum = 0.5) | no – skipped (skip_samples = 3) |
| 3–11   | bright flash | **yes** – 9 valid predictions |

The first `skip_samples = 3` frames are used as temporal context only.
All 9 valid predictions therefore fall inside the flash period.

### Flash luminance
```
flash_lum = mean_lum + contrast * (1 - mean_lum)
```
With `mean_lum = 0.5`:
- contrast = 0 → flash_lum = 0.5 (no flash, pure gray)
- contrast = 0.5 → flash_lum = 0.75
- contrast = 1.0 → flash_lum = 1.0 (maximum white)

### Load model

In [ ]:
from in_silico.model.mlflow_loader import ModelPaths, DataPaths, load_free_viewing_model_from_mlflow

model_paths = ModelPaths(
    checkpoint_uri="mlflow-artifacts:/621818231566971674/2f85fd6f5dda46e280456d3186618e1c/artifacts/6806be20120f307fa684cd4c637ad949_final.pth.tar",
    config_uri="mlflow-artifacts:/621818231566971674/2f85fd6f5dda46e280456d3186618e1c/artifacts/6806be20120f307fa684cd4c637ad949_final_cfg.pth.tar",
)

data_paths = DataPaths(session_dirs=["/mnt/data1/enigma/goliath_10_20_sandbox/37_3843837605846_0_V3A_V4/"])

out = load_free_viewing_model_from_mlflow(
    model_paths,
    data_paths,
    cuda_visible_devices="9",
    mlflow_tracking_uri="https://mlflow.enigmatic.stanford.edu/",
    mlflow_username="mlflow-runner",
    mlflow_password="x3i#U9*73N75",
)

In [ ]:
from in_silico.model.wrapper import ModelWrapper

model, skip_samples, cfg, extra = out

if skip_samples is None:
    skip_samples = cfg.trainer.skip_n_samples

KEY = "37_3843837605846_0_V3A_V4"
wrapper = ModelWrapper(model=model, skip_samples=skip_samples, key=KEY)
print(f"skip_samples = {skip_samples}")
print(f"session key  = {KEY}")
# With num_frames=12 and skip_samples=3, T_valid = 12 - 3 = 9 predicted frames.

### Select example neurons

We use the pre-saved V3A/V4 single-unit indices (good validation correlation).
A small subset is chosen for plotting.

In [ ]:
indices_v3a = np.load('/workdir/analysis_parametric/indices_v3a.npy')
print(f"{len(indices_v3a)} V3A/V4 neurons with good validation correlation")

# Pick a handful for plotting
EXAMPLE_NEURONS = indices_v3a[:8]
print(f"Example neuron indices: {EXAMPLE_NEURONS}")

---
## 1 – Full-field flash stimulus

The stimulus is a (T, 3, H, W) float32 array in [0, 1].
- **Pre-flash frames (0–2):** uniform gray at `mean_lum`.
- **Flash frames (3–11):** uniform bright at `mean_lum + contrast * (1 − mean_lum)`.

All three RGB channels are set to the same value (achromatic / luminance flash).

In [ ]:
def make_flash_frames(
    contrast: float = 1.0,
    mean_lum: float = 0.5,
    num_frames: int = 12,
    onset_frame: int = 3,
    H: int = 236,
    W: int = 420,
) -> np.ndarray:
    """
    Build an achromatic full-field flash stimulus.

    Args:
        contrast:    Flash contrast in [0, 1].
                     0 = no flash (all gray), 1 = maximum white.
        mean_lum:    Background (pre-flash) luminance in [0, 1].
        num_frames:  Total number of frames (model input length).
        onset_frame: Frame index at which the flash turns on.
        H, W:        Spatial dimensions matching the model input.

    Returns:
        frames: (num_frames, 3, H, W) float32 in [0, 1].
    """
    flash_lum = float(mean_lum) + float(contrast) * (1.0 - float(mean_lum))
    flash_lum = float(np.clip(flash_lum, 0.0, 1.0))

    frames = np.full((num_frames, 3, H, W), mean_lum, dtype=np.float32)
    frames[onset_frame:] = flash_lum
    return frames

In [ ]:
# Preview: show luminance of each frame for two contrasts
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

for ax, contrast in zip(axes, [0.5, 1.0]):
    frames = make_flash_frames(contrast=contrast)
    lum_per_frame = frames[:, 0, 0, 0]  # all pixels identical → just read one

    ax.step(np.arange(len(lum_per_frame)), lum_per_frame, where='post',
            color='steelblue', linewidth=2)
    ax.axvspan(0, 3, alpha=0.08, color='gray', label='skipped (context)')
    ax.axvspan(3, 12, alpha=0.08, color='gold', label='valid predictions')
    ax.set_ylim(-0.05, 1.1)
    ax.set_xlabel('Frame')
    ax.set_ylabel('Luminance')
    ax.set_title(f'Full-field flash  (contrast = {contrast})')
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# Visual preview of individual frames (contrast = 1)
frames_preview = make_flash_frames(contrast=1.0)

fig, axes = plt.subplots(2, 6, figsize=(18, 5))
for t, ax in enumerate(axes.flat):
    ax.imshow(frames_preview[t].transpose(1, 2, 0), vmin=0, vmax=1, cmap='gray')
    state = 'gray' if t < 3 else 'flash'
    ax.set_title(f't={t}  ({state})', fontsize=9)
    ax.axis('off')

plt.suptitle('Full-field flash frames  (contrast = 1.0, onset at frame 3)', y=1.02)
plt.tight_layout()
plt.show()

---
## 2 – Contrast sweep

We present the flash at several contrast levels and record the mean and
time-resolved predicted response of each example neuron.

**Prediction strategy:**
- One 12-frame window per contrast level.
- `wrapper.predict()` returns shape `(U, T_valid)` = `(653, 9)`.
- We average over the 9 valid time steps to get one scalar per neuron.

In [ ]:
CONTRASTS = np.linspace(-1, 1, 21)  # 21 steps: -1, -0.9, ..., 0, ..., 0.9, 1
MEAN_LUM = 0.5
ONSET_FRAME = 3
NUM_FRAMES = 12

# contrast_responses[i, u, t] = predicted response of neuron u at time t for contrasts[i]
contrast_responses = []  # list of (U, T_valid) arrays

for c in CONTRASTS:
    frames = make_flash_frames(contrast=c, mean_lum=MEAN_LUM,
                               num_frames=NUM_FRAMES, onset_frame=ONSET_FRAME)
    pred, _ = wrapper.predict(frames)   # (U, T_valid)
    contrast_responses.append(pred)

contrast_responses = np.stack(contrast_responses, axis=0)  # (n_contrasts, U, T_valid)
print(f"contrast_responses shape: {contrast_responses.shape}  → (n_contrasts, U, T_valid)")
print(f"contrast range: [{CONTRASTS[0]:.1f}, {CONTRASTS[-1]:.1f}]")

In [ ]:
# Time-resolved response for each example neuron at every contrast
T_valid = contrast_responses.shape[2]
time_axis = np.arange(skip_samples, skip_samples + T_valid) / 30.0  # seconds, 30 fps

# Diverging colormap: blue = dark flash (contrast < 0), red = bright flash (contrast > 0)
cmap = plt.cm.RdBu_r
norm = plt.Normalize(-1, 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
axes = axes.flat

for ax, unit_idx in zip(axes, EXAMPLE_NEURONS):
    for c in CONTRASTS:
        color = cmap(norm(c))
        ci = np.argmin(np.abs(CONTRASTS - c))
        ax.plot(time_axis, contrast_responses[ci, unit_idx, :],
                color=color, linewidth=1.2)
    ax.set_title(f'Neuron {unit_idx}', fontsize=10)
    ax.set_xlabel('Time in window (s)', fontsize=8)
    ax.set_ylabel('Predicted response', fontsize=8)
    sns.despine(ax=ax)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=list(axes), label='Flash contrast  (blue=dark, red=bright)',
             shrink=0.6, pad=0.02)

fig.suptitle('Time-resolved response to full-field flash – contrast sweep', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Mean response (averaged over valid time) vs contrast
mean_response = contrast_responses.mean(axis=2)  # (n_contrasts, U)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw
for unit_idx in EXAMPLE_NEURONS:
    axes[0].plot(CONTRASTS, mean_response[:, unit_idx], marker='o', markersize=3,
                 label=f'n{unit_idx}', linewidth=1.5)
axes[0].axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
axes[0].set_xlabel('Flash contrast')
axes[0].set_ylabel('Mean predicted response')
axes[0].set_title('Contrast–response curve (raw)')
axes[0].legend(fontsize=7, ncol=2)
sns.despine(ax=axes[0])

# Peak-normalised (per neuron, across contrasts)
peak = np.abs(mean_response).max(axis=0, keepdims=True)
peak = np.where(peak == 0, 1.0, peak)
mean_response_norm = mean_response / peak

for unit_idx in EXAMPLE_NEURONS:
    axes[1].plot(CONTRASTS, mean_response_norm[:, unit_idx], marker='o', markersize=3,
                 label=f'n{unit_idx}', linewidth=1.5)
axes[1].axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
axes[1].set_xlabel('Flash contrast')
axes[1].set_ylabel('Normalised response')
axes[1].set_title('Contrast–response curve (peak-normalised)')
axes[1].legend(fontsize=7, ncol=2)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.show()

---
## 3 – Gaze shift prior to flash onset

**Experimental design:**

| Frames | Visual stimulus | Gaze |
|--------|-----------------|------|
| 0 – (onset−1) | gray background | at `gaze_x_pre` (pre-saccade position) |
| onset – 11 | bright/dark flash | snapped to 0 (fixation at screen centre) |

The gaze step happens **before** the flash turns on.  All 9 valid predictions
fall inside the flash window, but the pre-flash context already encodes the
prior eye position.

Two plots are produced for a chosen **neuron** and a chosen **contrast**:
1. **Time course** – response over the 9 valid frames; one curve per gaze condition.
2. **Contrast–response curve** – mean response vs contrast; one curve per gaze condition.

The no-gaze condition (`gaze_x_pre = 0`) is drawn as a solid line; shifted
conditions are dashed.

In [ ]:
def make_gaze_step(
    gaze_x: float = 0.0,
    gaze_y: float = 0.0,
    num_frames: int = 12,
    onset_frame: int = 3,
) -> np.ndarray:
    """
    Pre-flash gaze step: eye is at (gaze_x, gaze_y) for frames [0, onset_frame),
    then snaps to (0, 0) at flash onset.

    Simulates a saccade that lands at screen centre exactly when the flash turns on.

    Returns:
        gaze_txy: (num_frames, 2) float32
    """
    gaze = np.zeros((num_frames, 2), dtype=np.float32)
    gaze[:onset_frame, 0] = float(gaze_x)
    gaze[:onset_frame, 1] = float(gaze_y)
    return gaze

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
UNIT_IDX      = indices_v3a[0]   # neuron to examine (change as needed)
PLOT_CONTRAST = 1.0              # contrast shown in the time-course panel

# Pre-flash gaze positions to compare (horizontal axis, px).
# 0 = fixation at centre (baseline); others = prior saccade amplitude.
GAZE_X_CONDITIONS = [0, 50, 100, 150, 200]

# ── Run contrast sweep for each gaze condition ────────────────────────────────
# gaze_contrast_resp[gx] → (n_contrasts, U, T_valid)
gaze_contrast_resp = {}

for gx in GAZE_X_CONDITIONS:
    gaze_txy = make_gaze_step(gaze_x=gx, onset_frame=ONSET_FRAME, num_frames=NUM_FRAMES)
    resp_g = []
    for c in CONTRASTS:
        frames = make_flash_frames(contrast=c, mean_lum=MEAN_LUM,
                                   num_frames=NUM_FRAMES, onset_frame=ONSET_FRAME)
        pred, _ = wrapper.predict(frames, gaze_txy=gaze_txy)  # (U, T_valid)
        resp_g.append(pred)
    gaze_contrast_resp[gx] = np.stack(resp_g, axis=0)  # (n_contrasts, U, T_valid)
    print(f"  gaze_x = {gx:+4.0f} px  done")

print(f"\nAll gaze conditions done.  Array shape per condition: {gaze_contrast_resp[0].shape}")
print(f"  → (n_contrasts={len(CONTRASTS)}, U=653, T_valid={gaze_contrast_resp[0].shape[2]})")

In [ ]:
T_valid     = gaze_contrast_resp[0].shape[2]
time_axis   = np.arange(skip_samples, skip_samples + T_valid) / 30.0  # s
contrast_idx = int(np.argmin(np.abs(CONTRASTS - PLOT_CONTRAST)))

# Colour palette: darker blue = larger gaze offset
gaze_colors = plt.cm.Blues(np.linspace(0.35, 0.9, len(GAZE_X_CONDITIONS)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Panel 1: time course at PLOT_CONTRAST ────────────────────────────────────
ax = axes[0]
for gx, color in zip(GAZE_X_CONDITIONS, gaze_colors):
    r    = gaze_contrast_resp[gx][contrast_idx, UNIT_IDX, :]
    ls   = '-' if gx == 0 else '--'
    lw   = 2.5 if gx == 0 else 1.8
    lbl  = f'gaze_x = {gx:+.0f} px' + (' (no shift)' if gx == 0 else '')
    ax.plot(time_axis, r, color=color, linewidth=lw, linestyle=ls, label=lbl)

ax.set_xlabel('Time in window (s)')
ax.set_ylabel('Predicted response')
ax.set_title(f'Time course  |  neuron {UNIT_IDX}  |  contrast = {PLOT_CONTRAST:+.1f}')
ax.legend(fontsize=8)
sns.despine(ax=ax)

# ── Panel 2: contrast–response curve ─────────────────────────────────────────
ax = axes[1]
for gx, color in zip(GAZE_X_CONDITIONS, gaze_colors):
    mean_r = gaze_contrast_resp[gx][:, UNIT_IDX, :].mean(axis=1)  # (n_contrasts,)
    ls     = '-' if gx == 0 else '--'
    lw     = 2.5 if gx == 0 else 1.8
    lbl    = f'gaze_x = {gx:+.0f} px' + (' (no shift)' if gx == 0 else '')
    ax.plot(CONTRASTS, mean_r, color=color, linewidth=lw, linestyle=ls,
            marker='o', markersize=3, label=lbl)

ax.axvline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax.set_xlabel('Flash contrast')
ax.set_ylabel('Mean predicted response')
ax.set_title(f'Contrast–response curve  |  neuron {UNIT_IDX}')
ax.legend(fontsize=8)
sns.despine(ax=ax)

plt.suptitle(
    f'Effect of pre-flash gaze shift  (flash onset = frame {ONSET_FRAME})',
    fontsize=12, y=1.02,
)
plt.tight_layout()
plt.show()

---
## 3 – Gaze sweep

The model receives gaze position as a `(T, 2)` float32 array in the same
coordinate system as the training data (eye-tracker coordinates in pixels).

- **`gaze_txy = np.zeros((T, 2))`** – fixation at the screen centre.
- **`gaze_txy[:, 0] = x_offset`** – shift gaze horizontally by `x_offset` pixels.
- **`gaze_txy[:, 1] = y_offset`** – shift gaze vertically by `y_offset` pixels.

The screen is 420 × 236 px (≈ 63 × 35 deg at 6.7 px/deg), so offsets of
±100 px correspond to ≈ ±15 deg — a large but plausible saccade.

We fix contrast at **1.0** and sweep gaze along the horizontal axis first,
then sweep 2-D positions on a coarser grid.

> **Note:** The gaze coordinate system depends on the preprocessing pipeline.
> If the training data normalises gaze to a different range, adjust the offsets below.

In [ ]:
GAZE_CONTRAST = 1.0   # contrast used throughout the gaze sweep
NUM_FRAMES = 12
ONSET_FRAME = 3

# Gaze offsets to sweep (pixels; adjust based on your coordinate system)
GAZE_X_VALUES = np.array([-200, -150, -100, -50, 0, 50, 100, 150, 200], dtype=np.float32)
GAZE_Y_VALUES = np.array([-100, -50, 0, 50, 100], dtype=np.float32)

flash_frames = make_flash_frames(contrast=GAZE_CONTRAST, num_frames=NUM_FRAMES, onset_frame=ONSET_FRAME)

In [ ]:
# --- Horizontal gaze sweep (gaze_y = 0) ---
# gaze_x_responses[i, u] = mean response of neuron u with gaze_x = GAZE_X_VALUES[i]

gaze_x_mean = []

for gx in GAZE_X_VALUES:
    gaze_txy = np.zeros((NUM_FRAMES, 2), dtype=np.float32)
    gaze_txy[:, 0] = gx
    pred, _ = wrapper.predict(flash_frames, gaze_txy=gaze_txy)   # (U, T_valid)
    gaze_x_mean.append(pred.mean(axis=1))                         # (U,)

gaze_x_mean = np.stack(gaze_x_mean, axis=0)  # (n_gaze_x, U)
print(f"gaze_x_mean shape: {gaze_x_mean.shape}  → (n_gaze_x, U)")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
axes = axes.flat

for ax, unit_idx in zip(axes, EXAMPLE_NEURONS):
    r = gaze_x_mean[:, unit_idx]
    ax.plot(GAZE_X_VALUES, r, color='steelblue', marker='o', markersize=4, linewidth=2)
    ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='center')
    ax.set_title(f'Neuron {unit_idx}', fontsize=10)
    ax.set_xlabel('Gaze X offset (px)', fontsize=8)
    ax.set_ylabel('Mean predicted response', fontsize=8)
    sns.despine(ax=ax)

fig.suptitle(
    f'Effect of horizontal gaze shift on predicted flash response'
    f'  (contrast = {GAZE_CONTRAST})',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2-D gaze sweep ---
# gaze_2d_mean[i, j, u] = mean response of neuron u with gaze = (GAZE_X_VALUES_2D[j], GAZE_Y_VALUES[i])

# Use a coarser grid for the 2-D sweep to keep runtime reasonable
GAZE_X_2D = np.array([-150, -75, 0, 75, 150], dtype=np.float32)
GAZE_Y_2D = np.array([-75, 0, 75], dtype=np.float32)

gaze_2d_mean = np.zeros((len(GAZE_Y_2D), len(GAZE_X_2D), 653), dtype=np.float32)

for i, gy in enumerate(GAZE_Y_2D):
    for j, gx in enumerate(GAZE_X_2D):
        gaze_txy = np.zeros((NUM_FRAMES, 2), dtype=np.float32)
        gaze_txy[:, 0] = gx
        gaze_txy[:, 1] = gy
        pred, _ = wrapper.predict(flash_frames, gaze_txy=gaze_txy)   # (U, T_valid)
        gaze_2d_mean[i, j, :] = pred.mean(axis=1)                    # (U,)

print(f"gaze_2d_mean shape: {gaze_2d_mean.shape}  → (n_gy, n_gx, U)")

In [ ]:
# Heatmap of mean response as a function of gaze position for each example neuron
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, unit_idx in zip(axes.flat, EXAMPLE_NEURONS):
    data = gaze_2d_mean[:, :, unit_idx]  # (n_gy, n_gx)
    im = ax.imshow(
        data,
        origin='upper',
        extent=[GAZE_X_2D[0], GAZE_X_2D[-1], GAZE_Y_2D[-1], GAZE_Y_2D[0]],
        aspect='auto',
        cmap='RdBu_r',
    )
    ax.scatter([0], [0], marker='+', color='white', s=80, zorder=5, label='centre')
    ax.set_title(f'Neuron {unit_idx}', fontsize=10)
    ax.set_xlabel('Gaze X (px)', fontsize=8)
    ax.set_ylabel('Gaze Y (px)', fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle(
    f'2-D gaze map – mean predicted flash response  (contrast = {GAZE_CONTRAST})',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()

---
## Summary

| Section | What was done |
|---------|---------------|
| 1 | Built an achromatic full-field flash with configurable contrast and onset frame |
| 2 | Swept contrast from 0 → 1; plotted time-resolved and mean contrast–response curves |
| 3 | Fixed contrast = 1, swept gaze along X (1-D) and across a 2-D grid; showed how gaze modulates the predicted flash response |

**Next steps:**
- Try a negative flash (bright → dark) by setting `flash_lum < mean_lum`.
- Combine gaze sweep with contrast sweep to map the full (contrast × gaze) space.
- Compare neurons with high vs low validation correlation.